### Import libraries 

In [2]:
import pandas as pd
import numpy as np 
from sklearn.ensemble import IsolationForest
import joblib

### Load Data 

In [7]:
employees = pd.read_csv('../data/employees.csv')
payroll = pd.read_csv('../data/payroll.csv')
dept = pd.read_csv('../data/departments.csv')

In [12]:
dept

,dept_id,dept_name
0,1,Sales
1,2,Engineering
2,3,HR
3,4,Finance
4,5,Operations


In [8]:
payroll

,payroll_id,employee_id,pay_period,gross_pay,deductions,net_pay,overtime_hours
0,1,1,2026-02-28,6147.79,1528.66,4619.13,10
1,2,2,2026-02-28,5279.71,872.66,4407.05,10
2,3,3,2026-02-28,6081.91,1025.75,5056.15,5
3,4,4,2026-02-28,6903.75,1232.39,5671.36,0
4,5,5,2026-02-28,4707.25,787.70,3919.55,0
...,...,...,...,...,...,...,...
496,497,497,2026-02-28,3703.42,921.33,2782.09,0
497,498,498,2026-02-28,4762.00,779.27,3982.73,0
498,499,499,2026-02-28,3905.42,857.30,3048.12,0
499,500,500,2026-02-28,3271.50,623.00,2648.50,0


In [9]:
employees

,employee_id,first_name,last_name,ssn,email,department_id,hire_date,salary,is_active
0,1,Harry,Mason,757-41-5089,codythompson@example.net,1.0,2026-02-04,67450,True
1,2,Carrie,Cortez,543-28-3579,joan05@example.com,1.0,2024-08-19,57926,True
2,3,Leah,Johnson,641-54-9311,debbie71@example.net,3.0,2024-08-10,69715,True
3,4,Steven,Tyler,658-03-5889,gibsonwilliam@example.com,2.0,2023-06-26,82845,True
4,5,Henry,Clark,730-07-8681,teresashepard@example.net,2.0,2022-11-15,56487,True
...,...,...,...,...,...,...,...,...,...
496,497,Jennifer,Davis,133-19-0028,elizabethmiller@example.org,4.0,2022-07-21,44441,True
497,498,Kelly,Hansen,194-51-8915,thudson@example.net,2.0,2022-12-02,57144,True
498,499,Jeff,Hampton,187-34-3519,uwebster@example.com,5.0,2023-06-30,46865,True
499,500,Marvin,Frederick,789-73-9813,alexander26@example.com,4.0,2023-02-24,39258,True


### Merge

In [14]:
df = payroll.merge(employees, on='employee_id', how='left')
df = df.merge(dept, left_on= 'department_id', right_on='dept_id', how='left')

,payroll_id,employee_id,pay_period,gross_pay,deductions,net_pay,overtime_hours,first_name,last_name,ssn,email,department_id,hire_date,salary,is_active,dept_id,dept_name
0,1,1,2026-02-28,6147.79,1528.66,4619.13,10,Harry,Mason,757-41-5089,codythompson@example.net,1.0,2026-02-04,67450,True,1.0,Sales
1,2,2,2026-02-28,5279.71,872.66,4407.05,10,Carrie,Cortez,543-28-3579,joan05@example.com,1.0,2024-08-19,57926,True,1.0,Sales
2,3,3,2026-02-28,6081.91,1025.75,5056.15,5,Leah,Johnson,641-54-9311,debbie71@example.net,3.0,2024-08-10,69715,True,3.0,HR
3,4,4,2026-02-28,6903.75,1232.39,5671.36,0,Steven,Tyler,658-03-5889,gibsonwilliam@example.com,2.0,2023-06-26,82845,True,2.0,Engineering
4,5,5,2026-02-28,4707.25,787.70,3919.55,0,Henry,Clark,730-07-8681,teresashepard@example.net,2.0,2022-11-15,56487,True,2.0,Engineering
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,497,497,2026-02-28,3703.42,921.33,2782.09,0,Jennifer,Davis,133-19-0028,elizabethmiller@example.org,4.0,2022-07-21,44441,True,4.0,Finance
497,498,498,2026-02-28,4762.00,779.27,3982.73,0,Kelly,Hansen,194-51-8915,thudson@example.net,2.0,2022-12-02,57144,True,2.0,Engineering
498,499,499,2026-02-28,3905.42,857.30,3048.12,0,Jeff,Hampton,187-34-3519,uwebster@example.com,5.0,2023-06-30,46865,True,5.0,Operations
499,500,500,2026-02-28,3271.50,623.00,2648.50,0,Marvin,Frederick,789-73-9813,alexander26@example.com,4.0,2023-02-24,39258,True,4.0,Finance


### Feature engineering

In [17]:
df['salary_monthly'] = df['salary'] / 12
df['net_pay_ratio'] = df['net_pay'] / df['gross_pay'] # should be <1, ~0.8
df['deductions_ratio'] = df['deductions'] / df['gross_pay']
df['overtime_ratio'] = df['overtime_hours'] / 160 # overtime hours as fraction of monthly hours
df['gross_vs_salary'] = df['gross_pay'] / df['salary_monthly'] # should be >=1 if overtime, else ~1

### Handle missing and infinite values 


In [22]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

### Model training  

In [25]:
features = ['gross_pay', 'deductions', 'net_pay', 'overtime_hours',
            'net_pay_ratio', 'deductions_ratio', 'overtime_ratio', 'gross_vs_salary']

X = df[features]

# Train Isolation Forest
model = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_score'] = model.fit_predict(X)
df['anomaly_score'] = df['anomaly_score'].map({1: 0, -1: 1})  # 1 = anomaly


# Add negative scores for easier sorting
df['anomaly_decision'] = model.decision_function(X)

In [26]:
extemely_Suspicion = df[df['anomaly_decision'] < 0]
Large_negative_number

,payroll_id,employee_id,pay_period,gross_pay,deductions,net_pay,overtime_hours,first_name,last_name,ssn,...,is_active,dept_id,dept_name,salary_monthly,net_pay_ratio,deductions_ratio,overtime_ratio,gross_vs_salary,anomaly_score,anomaly_decision
14,15,15,2026-02-28,3110.44,524.75,2585.70,10,Marie,Owens,667-19-6736,...,True,2.0,Engineering,2843.833333,0.831297,0.168706,0.06250,1.093749,1,-0.010696
73,74,74,2026-02-28,8259.95,2052.47,6207.49,20,Justin,Gentry,001-31-4523,...,True,2.0,Engineering,6955.750000,0.751517,0.248485,0.12500,1.187500,1,-0.073350
74,75,75,2026-02-28,2617.19,633.58,1983.61,5,Rebecca,Craig,676-18-8322,...,True,5.0,Operations,2500.000000,0.757916,0.242084,0.03125,1.046876,1,-0.014225
86,87,87,2026-02-28,7008.29,1060.42,5947.87,15,David,Williamson,744-28-5007,...,True,3.0,HR,6144.250000,0.848691,0.151309,0.09375,1.140626,1,-0.004887
110,111,111,2026-02-28,2723.45,436.87,2286.58,5,Elizabeth,Ross,610-39-8926,...,True,2.0,Engineering,2601.500000,0.839589,0.160411,0.03125,1.046877,1,-0.034834
121,122,122,2026-02-28,4587.61,807.15,3780.45,20,Lori,Kennedy,304-78-4111,...,True,4.0,Finance,3863.250000,0.824057,0.175941,0.12500,1.187500,1,-0.001126
125,126,126,2026-02-28,8826.16,1583.82,7242.33,15,Joan,Bailey,881-91-0815,...,True,5.0,Operations,7738.000000,0.820553,0.179446,0.09375,1.140625,1,-0.048964
130,131,131,2026-02-28,3635.73,643.04,2992.69,20,Jonathan,White,633-91-9711,...,True,5.0,Operations,3061.666667,0.823133,0.176867,0.12500,1.187500,1,-0.040590
153,154,154,2026-02-28,6282.17,961.92,5320.26,20,Joseph,Carroll,728-87-9522,...,True,1.0,Sales,5290.250000,0.846883,0.153119,0.12500,1.187500,1,-0.015698
173,174,174,2026-02-28,6698.23,1573.55,5124.69,25,Andrew,White,774-43-8262,...,True,5.0,Operations,5426.416667,0.765081,0.234920,0.15625,1.234374,1,-0.049953


### Save model 

In [ ]:
df.to_csv('data/payroll_with_anomalies.csv', index=False)
joblib.dump(model, 'models/isolation_forest.pkl')